In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
working_directory = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "src" / "procesado" / "limpiar_poblacion.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/procesado.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 1. Cleaning population

Read the Andalusian population Bronze table, apply the current cleaning steps, and preview the result.

In [3]:
from src.procesado.limpiar_poblacion import clean_population
from src.procesado.paths import (
    POPULATION_BRONZE_PATH,
    POPULATION_FAILED_PATH,
    POPULATION_SILVER_PATH,
)

population_bronze = pd.read_csv(
    POPULATION_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
population_cleaned = clean_population(population_bronze)

POPULATION_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
population_cleaned.to_parquet(POPULATION_SILVER_PATH, index=False)
population_failed = pd.read_parquet(POPULATION_FAILED_PATH)
print(
    f"Bronze rows: {len(population_bronze):,}; "
    f"cleaned rows: {len(population_cleaned):,}; "
    f"failed rows: {len(population_failed):,}; "
    f"saved to: {POPULATION_SILVER_PATH}"
)
print(f"Rejected rows saved to: {POPULATION_FAILED_PATH}")
display(population_cleaned.head(10))

Bronze rows: 1,201,050; cleaned rows: 3,925; failed rows: 0; saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\population_andalusia.parquet
Rejected rows saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\failed\population_failed.parquet


,municipality_code,municipality_name,province,province_code,year,population_total,population_18_50,source,downloaded_at_utc
0,04001,Abla,Almería,04,2021,1260,479,INE,2026-10-19 07:07:30.076995+00:00
1,04001,Abla,Almería,04,2022,1248,474,INE,2026-10-19 07:07:30.076995+00:00
2,04001,Abla,Almería,04,2023,1259,474,INE,2026-10-19 07:07:30.076995+00:00
3,04001,Abla,Almería,04,2024,1277,480,INE,2026-10-19 07:07:30.076995+00:00
4,04001,Abla,Almería,04,2025,1267,464,INE,2026-10-19 07:07:30.076995+00:00
5,04002,Abrucena,Almería,04,2021,1212,461,INE,2026-10-19 07:07:30.076995+00:00
6,04002,Abrucena,Almería,04,2022,1220,464,INE,2026-10-19 07:07:30.076995+00:00
7,04002,Abrucena,Almería,04,2023,1274,475,INE,2026-10-19 07:07:30.076995+00:00
8,04002,Abrucena,Almería,04,2024,1234,442,INE,2026-10-19 07:07:30.076995+00:00
9,04002,Abrucena,Almería,04,2025,1235,442,INE,2026-10-19 07:07:30.076995+00:00


# 2. Cleaning income

Read the INE Bronze table, keep per-person income, save the Silver Parquet, and preview the result.

In [ ]:
from src.procesado.limpiar_renta import clean_income
from src.procesado.paths import (
    INCOME_BRONZE_PATH,
    INCOME_FAILED_PATH,
    INCOME_SILVER_PATH,
)

income_bronze = pd.read_csv(
    INCOME_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
income_cleaned = clean_income(income_bronze)

INCOME_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
income_cleaned.to_parquet(INCOME_SILVER_PATH, index=False)
print(
    f"Bronze rows: {len(income_bronze):,}; "
    f"cleaned rows: {len(income_cleaned):,}; "
    f"failed rows: {len(pd.read_parquet(INCOME_FAILED_PATH)):,}; "
    f"saved to: {INCOME_SILVER_PATH}"
)
display(income_cleaned.head(10))

# 3. Cleaning gyms

Read the Andalusian OSM Bronze gym table, clean it, save the GeoParquet Silver table, and preview the result.

In [ ]:
from src.procesado.limpiar_gimnasios import clean_gyms
from src.procesado.paths import (
    GYMS_BRONZE_PATH,
    GYMS_FAILED_PATH,
    GYMS_SILVER_PATH,
)

gyms_bronze = pd.read_csv(
    GYMS_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
gyms_cleaned = clean_gyms(gyms_bronze)

GYMS_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
gyms_cleaned.to_parquet(GYMS_SILVER_PATH, index=False)
print(
    f"Bronze rows: {len(gyms_bronze):,}; "
    f"Silver rows: {len(gyms_cleaned):,}; "
    f"gyms: {gyms_cleaned['sport_centre_type'].eq('gym').sum():,}; "
    f"other sports facilities: "
    f"{gyms_cleaned['sport_centre_type'].eq('other_sport_facility').sum():,}; "
    f"failed rows: {len(pd.read_parquet(GYMS_FAILED_PATH)):,}"
)
print(f"Silver saved to: {GYMS_SILVER_PATH}")
display(gyms_cleaned.head(10))